In [1]:
import pathlib
import random

import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
import xarray as xr

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    BatchNormalization,
    Conv2D,
    MaxPooling2D,
    Conv2DTranspose,
    Reshape,
    Lambda,
    Cropping2D,
)
from tensorflow.keras.layers import (
    Activation,
    Dropout,
    Dense,
    Flatten,
    Input,
    UpSampling2D,
    Concatenate,
    ZeroPadding2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

In [2]:
tfrecord_dir = pathlib.Path('/gpfs/work2/0/prjs1027/tfrecords/run1')
output_variables = ["hs", "theta0_x", "theta0_y", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "ssh", "xcur", "ycur"]
boundary_variables = ["hs_bnd", "hswe_bnd", "tmm10_bnd"]
variable_fields = input_variables + output_variables + boundary_variables

min_max_values = xr.open_dataset(tfrecord_dir / "min-max-values/min-max-values.nc")

for bnd_var in boundary_variables:
    min_max_values[bnd_var] = min_max_values[bnd_var.split("_")[0]]

min_max_values["theta0_x"] = xr.DataArray(np.array([0., 1.]), dims='index')
min_max_values["theta0_y"] = xr.DataArray(np.array([0., 1.]), dims='index')

In [3]:
learning_rate = 1e-4
n_epochs = 2
batch_size = 64
steps_per_execution = 1

model_version = "swan_nn_v0015"

raster_shape = (481, 421, 2)
widths = [16, 32, 64, 128, 256, 256]
block_depth = 3

In [4]:
min_max_values

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/19)
    longitude  (index) float64 ...
    latitude   (index) float64 ...
    hs         (index) float64 ...
    botl       (index) float64 ...
    hswe       (index) float64 ...
    tmm10      (index) float64 ...
    ...         ...
    ywnd       (index) float64 ...
    hs_bnd     (index) float64 ...
    hswe_bnd   (index) float64 ...
    tmm10_bnd  (index) float64 ...
    theta0_x   (index) float64 0.0 1.0
    theta0_y   (index) float64 0.0 1.0

In [5]:
def get_files(data_path):
    files = tf.io.gfile.glob(data_path + "/" + "*.tfrecords")
    return files


def get_dataset(files):
    """return a tfrecord dataset with all tfrecord files"""
    dataset = tf.data.TFRecordDataset(files)
    dataset = dataset.map(tf_parse)
    return dataset


def tf_parse(eg, variable_fields=variable_fields):
    """parse an example (or batch of examples, not quite sure...)"""

    # here we re-specify our format
    # you can also infer the format from the data using tf.train.Example.FromString
    # but that did not work
    parse_dict =         {
            "height": tf.io.FixedLenFeature([], tf.int64),
            "width": tf.io.FixedLenFeature([], tf.int64),
            "depth": tf.io.FixedLenFeature([], tf.int64),
        }

    vars = variable_fields
    for var in vars:
        parse_dict[var] = tf.io.FixedLenFeature([], tf.string)

    example = tf.io.parse_example(
        eg[tf.newaxis],
        parse_dict,
    )

    variables = {}
    for var in variable_fields:
        variables[var] = tf.io.parse_tensor(example[var][0], out_type="float32")
        variables[var] = (variables[var] - min_max_values[var].values[0]) / (min_max_values[var].values[1] - min_max_values[var].values[0])
        variables[var] = tf.where(tf.math.is_nan(variables[var]), tf.zeros_like(variables[var]) - 1, variables[var])
        if var in output_variables:
            variables[var] = variables[var][..., -1]
            variables[var] = tf.reshape(variables[var], (481, 421, 1))


    variables["hs_bnd"] = tf.reshape(variables["hs_bnd"], (22, 1))
    variables["tmm10_bnd"] = tf.reshape(variables["tmm10_bnd"], (22, 1))
    variables["hswe_bnd"] = tf.reshape(variables["hswe_bnd"], (22, 1))
    boundary_inputs = tf.concat([variables["hs_bnd"], variables["tmm10_bnd"], variables["hswe_bnd"]], axis=-1)

    inputs = (variables["xwnd"], variables["ywnd"], variables["ssh"], variables["xcur"], variables["ycur"], boundary_inputs)
    outputs = (variables["hs"], variables["theta0_x"], variables["theta0_y"], variables["tmm10"], variables["tps"], variables["hswe"])
    
    # inputs = tf.io.parse_tensor(example["inputs"][0], out_type="float64")
    # inputs = tf.cast(inputs, tf.float32)
    # max_loc = tf.math.argmax(inputs)[0]

    # outputs = tf.io.parse_tensor(example["outputs"][0], out_type="float64")
    # outputs = tf.ensure_shape(outputs, raster_shape)

    # inputs = inputs / scale_input
    # outputs = tf.math.sign(outputs) * (tf.math.abs(outputs)) ** (1 / 6)
    # outputs = 10 * (2 * (outputs - output_min) / (output_max - output_min) - 1)
    # outputs = tf.cast(outputs, tf.float32)

    return inputs, outputs

In [6]:
# Get dataset and shuffle
random.seed(0)
files = get_files(str(tfrecord_dir))
random.shuffle(files)
dataset = get_dataset(files)
# Filter out all samples with high values
# dataset = dataset.filter(lambda x, y: tf.reduce_all(x <= 1))
# dataset = dataset.filter(lambda x, y: tf.reduce_all(y <= 10))

train_dataset = (
    dataset.shuffle(buffer_size=1000, seed=0, reshuffle_each_iteration=False)
    .take(int(0.7 * len(files) * 1000))
    .batch(batch_size, drop_remainder=True)
)
test_dataset = (
    dataset.shuffle(buffer_size=1000, seed=0, reshuffle_each_iteration=False)
    .skip(int(0.7 * len(files) * 1000))
    .batch(batch_size, drop_remainder=True)
)

2024-07-02 08:16:18.965652: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-07-02 08:16:18.965752: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (tcn99.local.snellius.surf.nl): /proc/driver/nvidia/version does not exist


In [11]:
for i, o in train_dataset.take(1):
    print(i[-1].shape)
    o = tf.keras.layers.Flatten()(i[-1])
    print(o)

(64, 22, 3)
tf.Tensor(
[[0.15052749 0.18973176 0.11389804 ... 0.11772213 0.20653805 0.09897628]
 [0.19364297 0.15620084 0.10472323 ... 0.3124572  0.19863334 0.24892905]
 [0.17416297 0.22078982 0.14577825 ... 0.3670225  0.23409441 0.3343419 ]
 ...
 [0.20632158 0.1853961  0.1412126  ... 0.28198186 0.22032182 0.24529548]
 [0.15052749 0.18973176 0.11389804 ... 0.11772213 0.20653805 0.09897628]
 [0.14725094 0.22616637 0.12942123 ... 0.31701303 0.24199109 0.29364854]], shape=(64, 66), dtype=float32)


In [7]:
def ResidualBlock(width, k):
    def apply(x):
        input_width = x.shape[-1]
        if input_width == width:
            residual = x
        else:
            residual = Conv2D(width, kernel_size=1)(x)

        x = BatchNormalization()(x)
        x = Conv2D(width, kernel_size=k, padding="same")(x)
        x = BatchNormalization()(x)
        x = Conv2D(width, kernel_size=k, padding="same")(x)

        x = tf.keras.layers.Add()([x, residual])

        return x

    return apply


def ResidualBlockTranspose(width, k):
    def apply(x):
        input_width = x.shape[-1]
        if input_width == width:
            residual = x
        else:
            residual = Conv2D(width, kernel_size=1)(x)

        x = BatchNormalization()(x)
        x = Conv2DTranspose(width, kernel_size=k, padding="same")(x)
        x = BatchNormalization()(x)
        x = Conv2DTranspose(width, kernel_size=k, padding="same")(x)

        x = tf.keras.layers.Add()([x, residual])

        return x

    return apply


def DownBlock(width, k, block_depth):
    def apply(x):
        # x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock(width, k)(x)
            # skips.append(x)
        # x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)
        x = tf.keras.layers.Conv2D(
            width, kernel_size=(3, 3), strides=2, padding="same"
        )(x)
        return x

    return apply


def UpBlock(width, k, block_depth):
    def apply(x):
        # x, skips = x
        x = tf.keras.layers.Conv2DTranspose(
            width, kernel_size=(3, 3), strides=2, padding="same"
        )(x)
        # x = UpSampling2D(size=2)(x)
        for _ in range(block_depth):
            # x = Concatenate()([x, skips.pop()])
            x = ResidualBlockTranspose(width, k)(x)
        return x

    return apply

In [8]:
def create_input_model(model_input, widths, k, block_depth):
    x = ZeroPadding2D(((15, 16), (45, 46)))(model_input)

    x = ResidualBlock(widths[0], k)(x)

    # for width in widths[1: -1]:
    #     x = DownBlock(width, k, block_depth)(x)

    return x

def create_output_model(x, widths, k, block_depth, name):
    # for width in reversed(widths[1: -1]):
    #     x = UpBlock(width, k, block_depth)(x)

    x = ResidualBlock(widths[0], k)(x)
    model_output = Cropping2D(((15, 16), (45, 46)))(x)

    model_output = Conv2DTranspose(1, kernel_size=1, padding="valid", activation="linear", name=name)(model_output)

    return model_output

In [9]:
def cnn_model(input_shape, bnd_input_shape, widths, k, block_depth):
    input_layers = {}
    network_inputs = {}
    for var in input_variables:
        input_layers[var] = Input(input_shape)
        network_inputs[var] = create_input_model(input_layers[var], widths, k, block_depth)

    # input_layers["bnd"] = Input(bnd_input_shape)
    # network
    
    x = tf.keras.layers.Add()([x[1] for x in network_inputs.items()])

    for width in widths[1: -1]:
        x = DownBlock(width, k, block_depth)(x)
    
    for _ in range(block_depth):
        x = ResidualBlock(widths[-1], k)(x)
    
    img_shape = K.int_shape(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    bnd_input = Input(bnd_input_shape)
    input_layers['bnd'] = bnd_input
    bnd_x = tf.keras.layers.Flatten()(bnd_input)
    bnd_x = tf.keras.layers.Dense(66)(bnd_x)
    bnd_x = tf.keras.layers.BatchNormalization()(bnd_x)
    bnd_x = tf.keras.layers.Dense(128)(bnd_x)
    bnd_x = tf.keras.layers.BatchNormalization()(bnd_x)
    bnd_x = tf.keras.layers.Dense(256)(bnd_x)
    bnd_x = tf.keras.layers.BatchNormalization()(bnd_x)
    bnd_x = tf.keras.layers.Dense(256)(bnd_x)
    bnd_x = tf.keras.layers.BatchNormalization()(bnd_x)
    
    x = tf.keras.layers.Concatenate()([x, bnd_x])
    x = tf.keras.layers.Dense(512)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(1024)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Dense(img_shape[1] * img_shape[2] * img_shape[3])(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Reshape((img_shape[1], img_shape[2], img_shape[3]))(x)
    

    for width in reversed(widths[1: -1]):
        x = UpBlock(width, k, block_depth)(x)
    
    # Thingies with the middle layer
    network_outputs = {}
    for var in output_variables:
        network_outputs[var] = create_output_model(x, widths, k, block_depth, name=var)
    
    # Create model
    model = Model(inputs=[x[1] for x in input_layers.items()], outputs=[y[1] for y in network_outputs.items()])

    return model

In [10]:
model = cnn_model(raster_shape, (22, 3), widths, 3, block_depth)
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=learning_rate, decay_steps=10000, decay_rate=0.9
)
opt = Adam(learning_rate=lr_schedule)

model.compile(
    loss="mean_squared_error", optimizer=opt, steps_per_execution=steps_per_execution
)

# log_dir = str(model_dir / ("logs/fit/" + model_version))
# tensorboard_callback = tf.keras.callbacks.TensorBoard(
#     log_dir=log_dir, histogram_freq=1
# )
# prediction_callback = PredictionHistory()
# debug_callback = DebugCallback()

# model.fit(
#     x=train_dataset,
#     validation_data=test_dataset,
#     epochs=n_epochs,
#     # callbacks=[tensorboard_callback],
#     verbose=2,
# )  # , prediction_callback])

# model.save(model_dir / (model_version + ".h5"))

In [11]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 481, 421, 2  0           []                               
                                )]                                                                
                                                                                                  
 input_2 (InputLayer)           [(None, 481, 421, 2  0           []                               
                                )]                                                                
                                                                                                  
 input_3 (InputLayer)           [(None, 481, 421, 2  0           []                               
                                )]                                                            

In [12]:
tf.keras.utils.plot_model(model, to_file='model.png', show_shapes=True)